In [3]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import glob
from pathlib import Path
from multiprocessing import Pool, cpu_count
from typing import List
import re
import math

In [3]:
def process_file(file: str) -> pd.DataFrame:
    file_path = Path(file)
    parts = file_path.name.split("_")
    chromosome = "".join(parts[1:3])  # e.g., ['chr10', '1'] → 'chr101'

    df = pd.read_csv(file_path, sep="\t", header=None)
    df.insert(0, "CHROM", chromosome)
    df.rename(columns={0: "START", 1: "END", 2: "RHO"}, inplace=True)
    return df


def read_files_parallel(files: str) -> List[pd.DataFrame]:
    file_names = glob.glob(files, recursive=True)
    with Pool(processes=cpu_count()//2) as pool:
        dfs = pool.map(process_file, file_names)
    return dfs

recombination_rate = read_files_parallel("../output/n50_*_W100_P20.rmap")


In [4]:
def make_windows(data_frame: pd.DataFrame, w_size: int) -> pd.DataFrame:
    raw_data = data_frame.to_numpy()
    midpoints = (raw_data[:, 1].astype(int) + raw_data[:, 2].astype(int)) // 2
    bin_ids = midpoints // w_size
    
    unique_bins, inverse_indeces = np.unique(bin_ids, return_inverse=True)
    binned_sums = np.zeros(len(unique_bins))
    binned_count = np.zeros(len(unique_bins))
    
    np.add.at(binned_sums, inverse_indeces, raw_data[:, 3].astype(float))
    np.add.at(binned_count, inverse_indeces, 1)
    
    binned_means = (binned_sums / binned_count) * 100 * 1e6
    starts = unique_bins * w_size
    ends = starts + w_size
    midpoint = (starts + ends) // 2
    chrom = raw_data[0, 0]
    results = np.column_stack([
        np.full(len(unique_bins), chrom),
        starts,
        ends,
        binned_means,
        midpoint
    ])
    return pd.DataFrame(results, columns=["chrom", "start", "end", "cM/Mb", "midpoint"]).astype({
        "chrom": str,
        "start": int,
        "end": int,
        "cM/Mb": float,
        "midpoint": int
    })


windows_5kb = [make_windows(df, 5000) for df in recombination_rate]
windows_10kb = [make_windows(df, 10000) for df in recombination_rate]        

In [44]:
def summarize_recombination_windows(windows: list[pd.DataFrame]) -> pd.DataFrame:
    summary_table = pd.DataFrame([
        {
            "chrom": df["chrom"].iloc[0],
            **pd.to_numeric(df["cM/Mb"], errors="coerce").describe()
        }
        for df in windows
    ])

    # Reorder and round columns
    columns = ["chrom", "count", "mean", "std", "min", "25%", "50%", "75%", "max"]
    summary_table = summary_table[columns]
    summary_table[columns[2:]] = summary_table[columns[2:]].round(3)

    # Sort chromosomes numerically if named chr1, chr2, ..., chrX, etc.
    summary_table = summary_table.sort_values(
        "chrom", 
        key=lambda x: x.str.extract(r'chr(\d+|W|Z)')[0].map(lambda s: int(s) if s and s.isdigit() else float('inf'))
    )

    return summary_table
summary_table = summarize_recombination_windows(windows_10kb)
summary_table.to_latex("recombination_rate_10kb_summary.tex", index=False, float_format="%.3f")

In [5]:
def chr_sort_key(df):
    # Ensure we're getting a string from the chrom column
    chrom = str(df.iloc[0]["chrom"]) if "chrom" in df.columns else str(df.iloc[0, 0])
    match = re.match(r"chr(\d+)", chrom)
    if match:
        return int(match.group(1))  # Numeric chromosomes
    else:
        return float("inf")         # Push non-numeric chromosomes (e.g., chrX) to the end

windows_5kb.sort(key=chr_sort_key)
windows_10kb.sort(key=chr_sort_key)

In [35]:
def save_transformed_data(datasets: List[pd.DataFrame]):
    for data in datasets:
        try:
            chrom_name = data["chrom"].iat[0]
            window_size = data["end"].iat[0] - data["start"].iat[0]
            filename = f"transformed_{chrom_name}_w{window_size}.tsv"
            data.to_csv(filename, sep="\t", header=True, index=False)
        except Exception as e:
            print(f"Failed to save: {data}")
            raise e

save_transformed_data(windows_10kb)

In [84]:
def plot_genome_wide_rho(windows: List[pd.DataFrame], plot_name: str | None = None) -> None:
    n = len(windows)
    cols = 1
    rows = n
    
    chrom_lengths = [df["midpoint"].max() - df["midpoint"].min() for df in windows]
    max_len = max(chrom_lengths)
    
    
    figure = make_subplots(rows=rows, 
                      cols=cols, 
                      subplot_titles=[f"Chromosome {i}" for i in range(1, 34)], 
                      shared_yaxes=True,
                      shared_xaxes=False)
    
    for i, df in enumerate(windows):
        tick_step = 10_000_000
        tickvals = list(range(0, max_len + tick_step, tick_step))
        ticktext = list(str(i // 10_000_000) for i in tickvals)
        row = i + 1
        
        figure.add_trace(
            go.Scatter(x = df["midpoint"].astype(int),
                       y = df["cM/Mb"].astype(float),
                       name = f"Chromosome {i}",
                       mode='lines',
                       line = dict(width=1, color="#722f37")),
            row,
            col = 1
        )
        
        figure.update_xaxes(
            title_text = "Genomic Position (Mb)",
            title_font = dict(size = 14, color = "black"),
            range=[0 - tick_step, max_len + tick_step],
            tickvals=tickvals,
            ticktext=ticktext,
            tickfont = dict(size=12, color='black'),
            ticks='outside',
            showline=False,
            linecolor='black',
            linewidth=2,
            row=row,
            col=1,
            gridcolor='white',
            showgrid=False
        )
        
    figure.update_layout(
        height=300 * rows,
        width=1900 * cols,
        title = {
            'text':'Recombination Rate over Genomic Positions',
            'x': 0.5,
            'y': 0.999,
            'yanchor': 'top',
            'xanchor': 'center',
            'font': dict(size=26, color = "black")
            },
        showlegend=False,
        margin=dict(t=80, l=20, r=20, b=20),
        plot_bgcolor='#f5f5f5'
    )
       
    figure.update_yaxes(
        title_text="cM/Mb",
        ticks='outside',
        tickfont=dict(size=12, color='black'),
        title_font = dict(size = 14, color = "black"),
        showline=False,
        linecolor='black',
        linewidth=2,
        gridcolor='white',
        showgrid=False)
    
    if plot_name:
        figure.write_html(plot_name, auto_open=True)
    figure.show()
    


plot_genome_wide_rho(windows_10kb, "Blackcap_recombination_rate_w10kb_n50.html")
        

In [43]:
def plot_rho_distribution(data: List[pd.DataFrame], plot_name: str | None = None):
    n = len(data)
    cols = 4
    rows = math.ceil(n/cols)
    figure = make_subplots(rows=rows,
                           cols=cols,
                           subplot_titles=[df["chrom"].iat[0] for df in data],
                           shared_yaxes=True)
    
    for i, df in enumerate(data):
        row = i // cols + 1
        col = i % cols + 1
        
        figure.add_trace(
            go.Histogram(x=df["cM/Mb"].astype(float), name=f"{df['chrom'].iat[0]}", showlegend=False, histnorm="probability"),
            row,
            col
        )
        
    figure.update_layout(
        height=300 * rows,
        width=300 * cols,
        title_text="Recombination Rate (cM/Mb) Distribution by Chromosome",
        bargap=0.2
    )
    figure.update_xaxes(title_text="cM/Mb")
    figure.update_yaxes(title_text="Frequency")
    
    if plot_name:
        try:
            figure.write_image(plot_name)
        except Exception as e:
            raise RuntimeError(f"Failed to save plot to {plot_name}: {e}")
    
    figure.show()

plot_rho_distribution(windows_10kb, "windows_10kb_n50_distribution.pdf")